In [4]:
!curl -fL https://github.com/VirtusLab/scala-cli/releases/latest/download/scala-cli-x86_64-pc-linux.gz | gzip -d > scala-cli
!chmod +x scala-cli
!./scala-cli version

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 42.5M  100 42.5M    0     0  25.2M      0  0:00:01  0:00:01 --:--:-- 32.3M
Scala CLI version: 1.18.0
Scala version (default): 3.9.0


In [5]:
%%writefile exceptions.sc
//> using scala 3

import scala.util.{Try, Success, Failure, Using}

// ---------- Custom exceptions ----------
class InvalidAgeException(msg: String) extends Exception(msg)            // checked-style
class InsufficientFundsException(val shortBy: Double)
    extends RuntimeException(s"Short by $shortBy")                        // unchecked

// ---------- Helper: run a block and classify what it throws ----------
def attempt(name: String)(block: => Any): Unit =
  try
    println(f"$name%-26s OK -> $block")
  catch
    case e: ArithmeticException | _: NumberFormatException =>
      println(f"$name%-26s ${e.getClass.getSimpleName}: ${e.getMessage}")
    case e: IndexOutOfBoundsException =>
      println(f"$name%-26s ${e.getClass.getSimpleName}: ${e.getMessage}")
    case e: RuntimeException =>                       // all other unchecked exceptions
      println(f"$name%-26s ${e.getClass.getSimpleName}: ${e.getMessage}")
    case e: java.io.IOException =>                    // checked in Java
      println(f"$name%-26s ${e.getClass.getSimpleName}: ${e.getMessage}")
    case e: Exception =>
      println(f"$name%-26s ${e.getClass.getSimpleName}: ${e.getMessage}")
    case e: Error =>                                  // JVM errors (not Exceptions)
      println(f"$name%-26s ${e.getClass.getSimpleName} (Error)")
  finally
    () // runs every time; use for cleanup

def recurse(n: Int): Int = 1 + recurse(n + 1)

val zero = 0
val nullStr: String = null
val arr = Array(1, 2, 3)
val anyVal: Any = "hello"

println("=== Built-in exceptions ===")
attempt("ArithmeticException")        { 10 / zero }
attempt("ArrayIndexOutOfBounds")      { arr(5) }
attempt("IndexOutOfBounds (List)")    { List(1, 2, 3)(5) }
attempt("NullPointerException")       { nullStr.length }
attempt("NumberFormatException")      { "abc".toInt }
attempt("ClassCastException")         { anyVal.asInstanceOf[Int] }
attempt("NoSuchElementException")     { List.empty[Int].head }
attempt("NoSuchElement (Map)")        { Map("a" -> 1)("b") }
attempt("UnsupportedOperation")       { Nil.tail }
attempt("IllegalArgumentException")   { require(false, "bad argument") }
attempt("IllegalStateException")      { throw new IllegalStateException("bad state") }
attempt("NegativeArraySize")          { new Array[Int](-1) }
attempt("MatchError")                 { (5: Any) match { case "a" => 1 } }
attempt("FileNotFoundException")      { new java.io.FileInputStream("missing.txt") }
attempt("StackOverflowError")         { recurse(0) }
attempt("AssertionError")             { assert(1 > 2, "math broke") }

println("\n=== Custom exceptions ===")
def setAge(age: Int): Int =
  if age < 0 then throw new InvalidAgeException(s"Age $age is invalid") else age

def withdraw(balance: Double, amount: Double): Double =
  if amount > balance then throw new InsufficientFundsException(amount - balance)
  else balance - amount

attempt("InvalidAgeException")        { setAge(-5) }
attempt("InsufficientFunds")          { withdraw(100, 150) }

println("\n=== try / catch / finally ===")
try
  println(10 / zero)
catch
  case e: ArithmeticException => println(s"Caught: ${e.getMessage}")
finally
  println("finally always runs")

println("\n=== Try (functional style) ===")
Try("42".toInt) match
  case Success(v)  => println(s"Parsed: $v")
  case Failure(e)  => println(s"Failed: ${e.getMessage}")

Try("xyz".toInt) match
  case Success(v)  => println(s"Parsed: $v")
  case Failure(e)  => println(s"Failed: ${e.getClass.getSimpleName}")

println(Try(10 / zero).getOrElse(-1))           // default on failure
println(Try(10 / zero).recover { case _: ArithmeticException => 0 })

println("\n=== Either ===")
def safeDivide(a: Int, b: Int): Either[String, Int] =
  if b == 0 then Left("Cannot divide by zero") else Right(a / b)

println(safeDivide(10, 2))
println(safeDivide(10, 0))

println("\n=== Using (auto-closes resources) ===")
val result = Using(scala.io.Source.fromString("line1\nline2")) { src =>
  src.getLines().toList
}
println(result)

println("\n=== Rethrow and wrap ===")


Overwriting exceptions.sc


In [6]:
!./scala-cli run exceptions.sc

Streaming output truncated to the last 5000 lines.
https://repo1.maven.org/maven2/org/scalameta/trees2_2.12/4.17.3/trees2_2.12-4.1…
    4.2% [          ] 512.0 KiB (1.4 MiB / s)
https://repo1.maven.org/maven2/org/scala-sbt/zinc-apiinfo_2.12/1.12.1/zinc-apii…
    2.2% [          ] 3.4 KiB (23.6 KiB / s)
https://repo1.maven.org/maven2/org/ow2/asm/asm/9.10.1/asm-9.10.1.jar
    0.0% [          ] 0B
https://repo1.maven.org/maven2/com/eed3si9n/gigahorse-core_2.12/0.9.4/gigahorse…
    0.0% [          ] 0B
https://repo1.maven.org/maven2/org/junit/platform/junit-platform-launcher/6.0.3…
    0.0% [          ] 0B
https://repo1.maven.org/maven2/org/scala-sbt/zinc-apiinfo_2.12/1.12.1/zinc-apii…
   46.4% [####      ] 72.0 KiB (431.1 KiB / s)
https://repo1.maven.org/maven2/org/scala-lang/scala-compiler/2.12.21/scala-comp…
    9.6% [          ] 1.0 MiB (1.9 MiB / s)
https://repo1.maven.org/maven2/org/scalameta/trees2_2.12/4.17.3/trees2_2.12-4.1…
    4.6% [          ] 560.0 KiB (1.5 MiB / s)
https://re

In [9]:
%%writefile BankAcc.sc

object BankAccount{
  def withdraw(
    balance : Double,
    amount : Double): Unit = {
      if(amount > balance)
        throw new Exception("Insufficient Funds")
    }
}

Overwriting BankAcc.sc


In [10]:
!./scala-cli run BankAcc.sc

Compiling project (Scala 3.9.0, JVM (17))
Compiled project (Scala 3.9.0, JVM (17))
